we train a simple Neural Network classifier on the same 5 input variables used in the fuzzy logic system. This model will be used in the Streamlit app as a second comparison against Mamdani and Sugeno

This does NOT replace the fuzzy system, it serves as a benchmark

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib

In [ ]:
df = pd.read_csv("../data/gtd_processed.csv")

prop_map = {1: 3, 2: 2, 3: 1, 4: 0}
df["prop_inverted"] = df["propextent"].map(prop_map)

X = df[["nkill", "nwound", "prop_inverted", "attack_encoded", "weapon_encoded"]].values
y = df["severity_index"].values

print(f"Total samples : {len(df):,}")
print(f"Class distribution:")
print(pd.Series(y).value_counts())

In [ ]:
# encode labels to integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Classes: {le.classes_}")

# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# save encoder and scaler for use in streamlit
joblib.dump(le, "../app/model/label_encoder.pkl")
joblib.dump(scaler, "../app/model/scaler.pkl")
print("Encoder and scaler saved.")

In [ ]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# train model

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# for testing after training, so it just load the model
# no need to train again

model = keras.models.load_model("../app/model/dl_model.keras")

print("Model loaded successfully")

In [ ]:
# plot training history

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history["accuracy"],     label="Train", color="#8e44ad")
axes[0].plot(history.history["val_accuracy"], label="Val",   color="#d35400")
axes[0].set_title("Accuracy per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"],     label="Train", color="#8e44ad")
axes[1].plot(history.history["val_loss"], label="Val",   color="#d35400")
axes[1].set_title("Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# eval

X_test32 = X_test.astype(np.float32)

y_pred = np.argmax(
    model(
        X_test32,
        training=False
    ).numpy(),
    axis=1
)

order_labels = ["Low", "Medium", "High", "Critical"]

y_test_labels = le.inverse_transform(y_test)
y_pred_labels = le.inverse_transform(y_pred)

acc = accuracy_score(y_test_labels, y_pred_labels)

print(f"Neural Network Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")

print(
    classification_report(
        y_test_labels,
        y_pred_labels,
        labels=order_labels,
        target_names=order_labels
    )
)

In [ ]:
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=order_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=order_labels)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Purples", colorbar=False)
plt.title("Neural Network Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
model.save("../app/model/dl_model.keras")
print("Model saved to app/model/dl_model.keras")

## Summary

- Neural Network trained on 5 input variables with 80/20 train-test split
- Architecture: Input(5) -> Dense(64) -> BatchNorm -> Dropout -> Dense(128) -> BatchNorm -> Dropout -> Dense(64) -> Dropout -> Output(4)
- Total parameters: 17,988 (70.27 KB) very lightweight model
- Trained for 30 epochs with batch size 256
- Neural Network achieved 96.73% accuracy on the test set
- Performance per class:
  - Low: precision 1.00, recall 1.00
  - Medium: precision 0.99, recall 0.80
  - High: precision 0.68, recall 0.99
  - Critical: precision 0.98, recall 0.98
- Training curve shows stable convergence with no significant overfitting
- Features scaled using StandardScaler before training
- Label encoder and scaler saved for use in Streamlit app
- Model saved to app/model/dl_model.keras
- Neural Network does NOT replace the fuzzy system, fuzzy remains the main system, DL serves as a benchmark comparison